# 05 — Feature Engineering

**Goal:** turn the EDA/statistical evidence into features, with a written justification —
and a leakage analysis — for every one. Logic lives in
[`src/feature_engineering.py`](../src/feature_engineering.py); this notebook explains,
demonstrates, and verifies it.

**The leakage-safety argument, stated once and precisely:** every transformation in
`engineer_features()` is *stateless and row-wise* — a row's features depend only on that row's
own values, never on any statistic computed across rows (no means, quantiles, or fitted
encoders). Such transforms are mathematically identical whether applied before or after the
train/test split, so applying them up front is safe. Everything that must be *fit* (one-hot
encoding, scaling) is deliberately deferred to the sklearn `Pipeline` in notebook 06, where it
sees training folds only. This split of responsibilities is the leakage-prevention design of the
whole project.

Each feature below answers the four required questions: **why useful → how calculated →
leakage? → production-appropriate?**


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.config import CLEAN_DATA_FILE
from src.feature_engineering import (
    CATEGORICAL_FEATURES, FEATURE_COLUMNS, FEATURES_DATA_FILE,
    NUMERIC_FEATURES, engineer_features,
)

df = pd.read_csv(CLEAN_DATA_FILE)
fe = engineer_features(df)
fe["churn01"] = (fe["Churn"] == "Yes").astype(int)

def churn_table(col):
    g = fe.groupby(col, observed=True)["churn01"].agg(rate="mean", n="count")
    g["rate"] = (g["rate"] * 100).round(1)
    return g

print(f"{fe.shape[0]:,} rows | {len(FEATURE_COLUMNS)} model features "
      f"({len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical)")

7,043 rows | 22 model features (4 numeric + 18 categorical)


## 1. `tenure_group` — lifecycle stage on fixed bins

- **Why useful:** churn falls monotonically with tenure (notebook 03), but the relationship is
  nonlinear (53% → 36% → … → 7%). Logistic regression sees numeric `tenure` linearly; the
  binned version lets it capture the early-lifecycle cliff. It's also the natural segmentation
  axis for SQL/business reporting.
- **How:** `pd.cut` on **fixed, a-priori edges** {6, 12, 24, 48} months, open-ended top bin.
- **Leakage?** No — edges are constants, not data-derived quantiles. (Quantile binning would
  leak fold statistics *and* shift between retrainings; fixed edges do neither.)
- **Production?** Yes — deterministic; the open "49+" bin absorbs any future tenure > 72.
- We keep numeric `tenure` too: trees use the raw value's full resolution, the linear model
  benefits from the bins; the redundancy is harmless and each model can use what serves it.

In [2]:
churn_table("tenure_group")

,rate,n
tenure_group,,
0-6,52.9,1481
13-24,28.7,1024
25-48,20.4,1594
49+,9.5,2239
7-12,35.9,705


## 2. `auto_pay` — manual vs automatic payment

- **Why useful:** the EDA's cleanest hidden structure: both automatic methods churn at 15–17%,
  both manual ones far higher (e-check 45.3%). The 4-level `PaymentMethod` forces models to
  rediscover this; the binary hands it to them — and gives the linear model one strong, readable
  coefficient.
- **How:** `PaymentMethod` contains "(automatic)" → Yes, else No.
- **Leakage?** No — a recoding of one attribute known at scoring time.
- **Production?** Yes — trivially derivable from any billing record.
- Raw `PaymentMethod` stays in the feature set (the e-check vs mailed-check distinction still
  carries signal); the pair costs little and lets regularization/trees arbitrate.

In [3]:
churn_table("auto_pay")

,rate,n
auto_pay,,
No,34.7,3977
Yes,16.0,3066


## 3. `num_protective` and `num_streaming` — service counts that respect the EDA

- **Why useful:** notebook 03 found an asymmetry: support/protection add-ons associate strongly
  with retention; streaming add-ons don't. *One* total count would average away exactly that
  finding, so we count the two groups separately (and skip the total, which would be their exact
  sum — pure collinearity). These also give models a compact "engagement depth" signal that
  individual Yes/No columns express only jointly.
- **How:** count of "Yes" across {OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport}
  (0–4) and {StreamingTV, StreamingMovies} (0–2). Treated as numeric — they're ordinal counts,
  and 5 levels don't warrant one-hot expansion.
- **Leakage?** No — row-wise arithmetic on service flags.
- **Production?** Yes.
- **Honest caveat (visible below):** `num_protective = 0` mixes no-internet customers (7.4%
  churn) with internet customers who declined every add-on (~40%+) — the count features work
  *alongside* `InternetService`, and tree models can express that interaction.

In [4]:
print(churn_table("num_protective").to_string())
print()
print(churn_table("num_streaming").to_string())

                rate     n
num_protective            
0               29.8  2793
1               38.9  1467
2               23.8  1372
3               12.4   941
4                5.3   470

               rate     n
num_streaming            
0              22.8  3544
1              31.4  1559
2              29.4  1940


## 4. Collapsing the structural levels

- **Problem:** "No internet service" appears in six add-on columns and "No phone service" in
  `MultipleLines` — verified in notebook 01 to be *exactly* redundant with
  `InternetService`/`PhoneService`.
- **Action:** collapse to "No" in those seven columns. No information is lost —
  `InternetService` and `PhoneService` remain as features carrying it once.
- **Why it matters:** removes 7 one-hot columns that would be perfectly collinear with existing
  ones — smaller model space, cleaner coefficients, tidier SHAP output later.

In [5]:
print("MultipleLines levels:", sorted(fe["MultipleLines"].unique()))
print("TechSupport levels:  ", sorted(fe["TechSupport"].unique()))

MultipleLines levels: ['No', 'Yes']
TechSupport levels:   ['No', 'Yes']


## 5. Features **rejected** — with the evidence

Testing and discarding candidates is as much a part of feature engineering as inventing them.

**`avg_charge_per_month` (= TotalCharges / tenure):** the "average monthly revenue proxy" is
mathematically well-defined (with a tenure-0 guard), but measured against the data it correlates
**r = 0.996** with `MonthlyCharges` — it is the same feature wearing a hat. Rejected as pure
redundancy.

**`charge_growth` (= MonthlyCharges − avg_charge_per_month):** the plausible story — "customers
whose bills rose above their lifetime average are churn risks" — dies on contact with the data:
correlation with churn **0.002**, and churn across its quartiles is non-monotonic noise
(28.5% / 30.0% / 16.7% / 28.8%). Rejected for having no detectable signal.

**Total add-on count:** exact sum of the two kept counts. Rejected as perfect collinearity.

**`TotalCharges` itself (dropped from the feature list):** r = 0.83 with tenure, 0.65 with
MonthlyCharges (notebook 03) — and the *only* increment it offers beyond {tenure ×
MonthlyCharges} is precisely the `charge_growth` quantity just shown to carry nothing. Dropping
it costs ≈ no information and buys interpretable coefficients. (It stays in the saved dataset
for SQL/reporting — it's a real business quantity; it just doesn't enter the model.)

**`customerID` (excluded):** a random key; at best noise, at worst a memorization handle for
tree models.

In [6]:
check = fe[["MonthlyCharges"]].assign(
    avg_charge=(fe["TotalCharges"] / fe["tenure"].replace(0, 1)).where(fe["tenure"] > 0, fe["MonthlyCharges"]),
)
check["charge_growth"] = check["MonthlyCharges"] - check["avg_charge"]
print(f'corr(avg_charge, MonthlyCharges) = {check["avg_charge"].corr(check["MonthlyCharges"]):.4f}')
print(f'corr(charge_growth, churn)       = {check["charge_growth"].corr(fe["churn01"]):.3f}')

corr(avg_charge, MonthlyCharges) = 0.9962
corr(charge_growth, churn)       = 0.002


## 6. Verify the leakage-safety property in code

The stateless claim is testable: engineering a *subset* of rows must give byte-identical results
to engineering everything and then subsetting. A cross-row statistic anywhere would break this.

In [7]:
subset_first = engineer_features(df.head(100))
all_then_subset = engineer_features(df).head(100)
pd.testing.assert_frame_equal(subset_first, all_then_subset)
assert len(fe) == len(df), "row count preserved"
assert fe[FEATURE_COLUMNS].isna().sum().sum() == 0, "no NaNs in model features"
print("PASS: row-wise/stateless property verified; row count preserved; no NaNs in features")

PASS: row-wise/stateless property verified; row count preserved; no NaNs in features


## 7. Save the feature dataset

In [8]:
out = engineer_features(df)
out.to_csv(FEATURES_DATA_FILE, index=False)
print(f"Saved {out.shape[0]:,} rows x {out.shape[1]} cols -> {FEATURES_DATA_FILE.name}")
print(f"\nModel feature set ({len(FEATURE_COLUMNS)}):")
print("  numeric:    ", NUMERIC_FEATURES)
print("  categorical:", CATEGORICAL_FEATURES)

Saved 7,043 rows x 25 cols -> telco_churn_features.csv

Model feature set (22):
  numeric:     ['tenure', 'MonthlyCharges', 'num_protective', 'num_streaming']
  categorical: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group', 'auto_pay']


## Summary

| Feature | Evidence | Status |
|---|---|---|
| `tenure_group` (fixed bins) | churn 52.9% → 9.5% across groups | **added** |
| `auto_pay` | 16.0% vs 34.7% churn | **added** |
| `num_protective` (0–4) | 5.3% churn at 4 vs ~30–39% at 0–1 | **added** |
| `num_streaming` (0–2) | weak but distinct from protective | **added** |
| structural levels collapsed | exactly redundant (notebook 01) | **simplified** |
| `avg_charge_per_month` | r = 0.996 with MonthlyCharges | rejected |
| `charge_growth` | churn corr 0.002 | rejected |
| total add-on count | exact sum of kept counts | rejected |
| `TotalCharges` | r = 0.83 tenure; unique part ≈ noise | dropped from model |
| `customerID` | random key | excluded |

**22 model features** (4 numeric + 18 categorical), all stateless-derivable for any new
customer. Encoding and scaling intentionally deferred to the modeling pipeline (notebook 06).
